# 📈 CloudSync SaaS Analytics: 02. Exploratory Data Analysis & Executive KPIs

**Project:** CloudSync Customer Churn & Retention Analytics  
**Dataset:** CloudSync SaaS Cleaned Multi-Table Repository  
**Objective:** Calculate foundational SaaS executive KPIs (MRR, ARR, ARPU, LTV, Churn Rate, Retention Rate), evaluate univariate/bivariate distributions, analyze cohort retention dynamics, and diagnose revenue loss drivers.

---

### Core SaaS KPI Formulas Implemented:
- $\text{MRR} = \sum \text{monthly\_fee}$ for all Active subscriptions
- $\text{ARR} = \text{MRR} \times 12$
- $\text{ARPU} = \frac{\text{MRR}}{\text{Active Customers}}$
- $\text{Customer Churn Rate (\%)} = \frac{\text{Churned Customers}}{\text{Total Customers}} \times 100$
- $\text{Customer Retention Rate (\%)} = 100 - \text{Churn Rate}$
- $\text{Customer Lifetime Value (LTV)} = \text{ARPU} \times \text{Average Active Tenure (Months)}$
- $\text{Annualized Revenue Lost to Churn} = \sum \text{Churned Monthly Fees} \times 12$

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 300

CLEAN_DIR = '../data/cleaned'

df_cust = pd.read_csv(os.path.join(CLEAN_DIR, "customers_clean.csv"), parse_dates=['signup_date'])
df_subs = pd.read_csv(os.path.join(CLEAN_DIR, "subscriptions_clean.csv"), parse_dates=['subscription_start_date', 'subscription_end_date'])
df_act = pd.read_csv(os.path.join(CLEAN_DIR, "customer_activity_clean.csv"), parse_dates=['activity_month'])
df_tickets = pd.read_csv(os.path.join(CLEAN_DIR, "support_tickets_clean.csv"), parse_dates=['ticket_date'])
df_payments = pd.read_csv(os.path.join(CLEAN_DIR, "payments_clean.csv"), parse_dates=['payment_date'])
df_fb = pd.read_csv(os.path.join(CLEAN_DIR, "customer_feedback_clean.csv"), parse_dates=['feedback_date'])

# Unified analysis master
CURRENT_DT = pd.to_datetime("2024-12-31")
df_master = df_cust.merge(df_subs, on='customer_id', how='inner')
df_master['effective_end_date'] = df_master['subscription_end_date'].fillna(CURRENT_DT)
df_master['tenure_months'] = ((df_master['effective_end_date'] - df_master['subscription_start_date']).dt.days / 30.4).round(1)

print("Unified dataset ready for exploratory analysis.")

## 1. Executive SaaS KPI Scorecard Calculation

In [ ]:
total_cust = len(df_cust)
active_cust = len(df_subs[df_subs['subscription_status'] == 'Active'])
churned_cust = len(df_subs[df_subs['subscription_status'] == 'Churned'])
paused_cust = len(df_subs[df_subs['subscription_status'] == 'Paused'])

churn_rate = (churned_cust / total_cust) * 100
retention_rate = ((total_cust - churned_cust) / total_cust) * 100

mrr = df_subs[df_subs['subscription_status'] == 'Active']['monthly_fee'].sum()
arr = mrr * 12
arpu = mrr / active_cust if active_cust > 0 else 0

avg_tenure = df_master['tenure_months'].mean()
avg_tenure_act = df_master[df_master['subscription_status'] == 'Active']['tenure_months'].mean()
avg_tenure_churn = df_master[df_master['subscription_status'] == 'Churned']['tenure_months'].mean()

cltv = arpu * avg_tenure_act
monthly_lost = df_subs[df_subs['subscription_status'] == 'Churned']['monthly_fee'].sum()
arr_lost = monthly_lost * 12

avg_nps = df_fb['nps_score'].mean()
avg_csat = df_fb['satisfaction_score'].mean()

kpi_summary = pd.DataFrame([
    {"KPI Metric": "Total Customers", "Value": f"{total_cust:,}"},
    {"KPI Metric": "Active Customers", "Value": f"{active_cust:,}"},
    {"KPI Metric": "Churned Customers", "Value": f"{churned_cust:,} ({churn_rate:.2f}%)"},
    {"KPI Metric": "Customer Retention Rate", "Value": f"{retention_rate:.2f}%"},
    {"KPI Metric": "Monthly Recurring Revenue (MRR)", "Value": f"${mrr:,.2f}"},
    {"KPI Metric": "Annual Recurring Revenue (ARR)", "Value": f"${arr:,.2f}"},
    {"KPI Metric": "Average Revenue Per User (ARPU)", "Value": f"${arpu:.2f}"},
    {"KPI Metric": "Customer Lifetime Value (LTV)", "Value": f"${cltv:,.2f}"},
    {"KPI Metric": "Average Active Customer Tenure", "Value": f"{avg_tenure_act:.1f} months"},
    {"KPI Metric": "Average Churned Customer Tenure", "Value": f"{avg_tenure_churn:.1f} months"},
    {"KPI Metric": "Monthly Revenue Lost to Churn", "Value": f"${monthly_lost:,.2f}"},
    {"KPI Metric": "Annualized Revenue Lost to Churn", "Value": f"${arr_lost:,.2f}"},
    {"KPI Metric": "Average Net Promoter Score (NPS)", "Value": f"{avg_nps:.2f} / 10"},
    {"KPI Metric": "Average Customer Satisfaction (CSAT)", "Value": f"{avg_csat:.2f} / 5"}
])

kpi_summary

## 2. Churn Breakdown by Business Dimension

In [ ]:
# 1. Churn by Contract Type
churn_contract = df_master.groupby('contract_type').agg(
    total_users=('customer_id', 'count'),
    churned_users=('subscription_status', lambda x: (x == 'Churned').sum()),
    churn_rate=('subscription_status', lambda x: (x == 'Churned').mean() * 100)
).reset_index()

# 2. Churn by Subscription Plan
churn_plan = df_master.groupby('plan_name').agg(
    total_users=('customer_id', 'count'),
    churned_users=('subscription_status', lambda x: (x == 'Churned').sum()),
    churn_rate=('subscription_status', lambda x: (x == 'Churned').mean() * 100)
).reset_index()

# 3. Churn by Customer Segment
churn_seg = df_master.groupby('customer_segment').agg(
    total_users=('customer_id', 'count'),
    churned_users=('subscription_status', lambda x: (x == 'Churned').sum()),
    churn_rate=('subscription_status', lambda x: (x == 'Churned').mean() * 100)
).reset_index()

print("--- Churn by Contract Type ---")
display(churn_contract)
print("
--- Churn by Plan ---")
display(churn_plan)
print("
--- Churn by Segment ---")
display(churn_seg)

## 3. Visualizing Core SaaS Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Churn by Contract Type Bar Chart
sns.barplot(data=churn_contract, x='contract_type', y='churn_rate', ax=axes[0, 0], color='#2c7bb6')
axes[0, 0].set_title('Churn Rate by Contract Type (%)', fontweight='bold', fontsize=12)
axes[0, 0].set_ylabel('Churn Rate (%)')
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.8), ha='center', fontweight='bold')

# 2. Churn by Plan
sns.barplot(data=churn_plan, x='plan_name', y='churn_rate', ax=axes[0, 1], color='#8e44ad', order=['Basic', 'Professional', 'Business', 'Enterprise'])
axes[0, 1].set_title('Churn Rate by Plan Tier (%)', fontweight='bold', fontsize=12)
axes[0, 1].set_ylabel('Churn Rate (%)')
for p in axes[0, 1].patches:
    axes[0, 1].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.8), ha='center', fontweight='bold')

# 3. Churn by Acquisition Channel
chan_churn = df_master.groupby('acquisition_channel')['subscription_status'].apply(lambda x: (x == 'Churned').mean() * 100).sort_values(ascending=False).reset_index()
sns.barplot(data=chan_churn, x='subscription_status', y='acquisition_channel', ax=axes[1, 0], color='#e67e22')
axes[1, 0].set_title('Churn Rate by Acquisition Channel (%)', fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel('Churn Rate (%)')

# 4. Tenure KDE
sns.kdeplot(data=df_master[df_master['subscription_status']=='Active']['tenure_months'], label='Active Users', shade=True, color='#27ae60', ax=axes[1, 1])
sns.kdeplot(data=df_master[df_master['subscription_status']=='Churned']['tenure_months'], label='Churned Users', shade=True, color='#e74c3c', ax=axes[1, 1])
axes[1, 1].set_title('Customer Tenure Density (Months)', fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel('Tenure (Months)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Cohort Retention Matrix Analysis

In [ ]:
df_master['cohort_quarter'] = df_master['subscription_start_date'].dt.to_period('Q').astype(str)
cohort_pivot = df_master.groupby(['cohort_quarter', 'subscription_status'])['customer_id'].count().unstack(fill_value=0)
cohort_pivot['Total'] = cohort_pivot.sum(axis=1)
cohort_pivot['Retention_Rate_%'] = (cohort_pivot.get('Active', 0) / cohort_pivot['Total'] * 100).round(2)
cohort_pivot['Churn_Rate_%'] = (cohort_pivot.get('Churned', 0) / cohort_pivot['Total'] * 100).round(2)

display(cohort_pivot[['Total', 'Active', 'Churned', 'Retention_Rate_%', 'Churn_Rate_%']])